# Evaluation & Verification

This notebook verifies that all retrieval system components are functional and properly integrated. Both BM25 and semantic search indexes are tested on sample queries, and evaluation documentation is generated. The notebook confirms completion of the retrieval system foundation.

**Workflow:**

1. Verify artifacts exist
2. Load retrieval indexes
3. Test on queries
4. Generate evaluation document
5. Prepare submission

**Note:** Run notebooks in order: `01_exploration.ipynb`, `02_data_preparation.ipynb`, `03_bm25_keyword_search.ipynb`, `04_semantic_embedding_search.ipynb`, then `05_evaluation_and_verification.ipynb`.


## Setup: Initialize Paths

Set up project paths for the entire notebook.

In [1]:
from pathlib import Path
import sys

# Get project root (notebook is in notebooks/ folder)
notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Add project root to path
sys.path.insert(0, str(project_root))

print(f" Project root: {project_root}")
print(f" Paths initialized for all cells below")

 Project root: /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki
 Paths initialized for all cells below


## 1. Verify Artifacts

Check that all required files from data exploration and retrieval implementation exist.

In [2]:
import os
from pathlib import Path

# Get project root
notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Define all paths using project_root
checks = {
    "data/raw/ files": len([f for f in os.listdir(project_root / 'data' / 'raw') if f.endswith('.gz')]) > 0 if os.path.isdir(project_root / 'data' / 'raw') else False,
    "data/processed/books_sample.parquet": os.path.exists(project_root / 'data' / 'processed' / 'books_sample.parquet'),
    "data/processed/corpus.pkl": os.path.exists(project_root / 'data' / 'processed' / 'corpus.pkl'),
    "src/utils.py": os.path.exists(project_root / 'src' / 'utils.py'),
    "src/bm25.py": os.path.exists(project_root / 'src' / 'bm25.py'),
    "data/processed/bm25_index.pkl": os.path.exists(project_root / 'data' / 'processed' / 'bm25_index.pkl'),
    "src/semantic.py": os.path.exists(project_root / 'src' / 'semantic.py'),
    "data/processed/semantic_index": os.path.exists(project_root / 'data' / 'processed' / 'semantic_index'),
}

completed = sum(1 for v in checks.values() if v)
print(f"Verification: {completed}/{len(checks)} components")
print()
for name, result in checks.items():
    status = " OK" if result else " MISSING"
    print(f"  {status}: {name}")

Verification: 8/8 components

   OK: data/raw/ files
   OK: data/processed/books_sample.parquet
   OK: data/processed/corpus.pkl
   OK: src/utils.py
   OK: src/bm25.py
   OK: data/processed/bm25_index.pkl
   OK: src/semantic.py
   OK: data/processed/semantic_index


## 2. Test BM25 Retriever

Load BM25 index and execute comparative tests on sample queries.

In [3]:
from pathlib import Path
import sys
import pandas as pd

# Get project root
notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

sys.path.insert(0, str(project_root))

from src.bm25 import BM25Retriever

# Use absolute paths
bm25_index_file = str(project_root / 'data' / 'processed' / 'bm25_index.pkl')

print("Loading BM25 retriever...")
bm25 = BM25Retriever()
bm25.load(bm25_index_file)
print(" BM25 loaded successfully\n")

# Load data for result enrichment
df = pd.read_parquet(project_root / 'data' / 'processed' / 'books_sample.parquet')

# 10 queries spanning easy (keyword), medium (semantic), and complex difficulty
test_queries = [
    # Easy - keyword-based
    "mystery novel",
    "cookbook recipes",
    "science fiction space",
    "python programming",
    # Medium - semantic
    "book to help with anxiety",
    "story about finding yourself",
    "guide for first time parents",
    # Complex
    "best book to learn machine learning with no math background",
    "historical fiction set in world war 2 from a female perspective",
    "self help book for overcoming procrastination and building better habits",
]

print("="*60)
print("BM25 Test Results")
print("="*60)

for query in test_queries:
    results = bm25.search(query, top_k=5)
    
    print(f"\nQuery: '{query}'")
    for rank, (doc_id, score) in enumerate(results, 1):
        book_title = df.iloc[doc_id]['product_title']
        print(f"  {rank}. Doc #{doc_id} - {book_title} - Score: {score:.2f}")

print("\n BM25 retriever working correctly!")

Loading BM25 retriever...
BM25 index loaded from /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/bm25_index.pkl
 BM25 loaded successfully

BM25 Test Results

Query: 'mystery novel'
  1. Doc #39571 - The Woman in the Window: A Novel - Score: 8.58
  2. Doc #63216 - One Little Wish: a romantic suspense novel (The Little Things Mystery Series) - Score: 8.56
  3. Doc #47298 - The Last Lie Told (Finley O’Sullivan Book 1) - Score: 8.37
  4. Doc #517 - Code Name: Camelot - An Action Thriller Novel (A Noah Wolf Novel, Thriller, Action, Mystery Book 1) - Score: 8.36
  5. Doc #29208 - Mystery at Seagrave Hall: A totally addictive cozy mystery novel (An Eve Mallow Mystery Book 3) - Score: 8.36

Query: 'cookbook recipes'
  1. Doc #74371 - Japanese Cookbook for Beginners: 2 Books in 1, Sushi Cookbook + Ramen Cookbook, Quick and Easy Japanese Recipes to Make a Perfect Dinner at Home (Maggie Barton's Recipe Books) - Score: 14.78
  2. Doc #14190 - Tacos: Recipes and Pr

## 3. Test Semantic Retriever

Load semantic search index and test on sample queries to verify meaning-based document retrieval.



In [4]:
from pathlib import Path
import sys
import pandas as pd

notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

sys.path.insert(0, str(project_root))

from src.semantic import SemanticRetriever

semantic_index_dir = project_root / 'data' / 'processed' / 'semantic_index'
semantic_index_file = str(semantic_index_dir / 'faiss_index')

print("Loading Semantic retriever...")
semantic = SemanticRetriever()
semantic.load(semantic_index_file)
print(" Semantic loaded successfully\n")

# Load data for result enrichment
df = pd.read_parquet(project_root / 'data' / 'processed' / 'books_sample.parquet')

# Same 10 queries as BM25 cell above
test_queries = [
    # Easy - keyword-based
    "mystery novel",
    "cookbook recipes",
    "science fiction space",
    "python programming",
    # Medium - semantic
    "book to help with anxiety",
    "story about finding yourself",
    "guide for first time parents",
    # Complex
    "best book to learn machine learning with no math background",
    "historical fiction set in world war 2 from a female perspective",
    "self help book for overcoming procrastination and building better habits",
]

print("="*60)
print("Semantic Test Results")
print("="*60)

for query in test_queries:
    results = semantic.search(query, top_k=5)
    
    print(f"\nQuery: '{query}'")
    for rank, (doc_id, score) in enumerate(results, 1):
        book_title = df.iloc[doc_id]['product_title']
        print(f"  {rank}. Doc #{doc_id} - {book_title} - Score: {score:.2f}")

print("\n Semantic retriever working correctly!")

Loading Semantic retriever...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index loaded from /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/semantic_index/faiss_index
 Semantic loaded successfully

Semantic Test Results

Query: 'mystery novel'
  1. Doc #42383 - And Then There Were None - Score: 0.68
  2. Doc #23725 - Mystery: An Alex Delaware Novel - Score: 0.70
  3. Doc #72439 - The Stranger Diaries - Score: 0.72
  4. Doc #17947 - The Anonymous Source - Score: 0.72
  5. Doc #85997 - The Lost Spy (Slim Moran Mysteries Book 1) - Score: 0.72

Query: 'cookbook recipes'
  1. Doc #89789 - The Complete Cooking for Two Cookbook: 650 Recipes for Everything You'll Ever Want to Make (The Complete ATK Cookbook Series) - Score: 0.39
  2. Doc #53268 - The Perfect Guide for New Cooks (Love Food) - Score: 0.44
  3. Doc #24351 - Recipe Journal: Blank Cookbook To Write In - Paperback (Blank Cookbooks and Recipe Books) - Score: 0.45
  4. Doc #79230 - Blank Recipe Book: Recipe Journal ( Gifts for Foodies / Cooks / Chefs / Cooking ) [ Sof

## 4. Results Summary

Top-5 retrieval results for all 10 queries across BM25 and Semantic search. Difficulty levels: **E** = Easy (keyword), **M** = Medium (semantic), **C** = Complex.

---

### Query 1 [E]: "mystery novel"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | Trust Me | 9.19 | The Da Vinci Code | 0.51 |
| 2 | Disappearance of Adèle Bedeau | 8.95 | The Missing American | 0.67 |
| 3 | Seasons | 8.94 | In Peppermint Peril | 0.68 |
| 4 | The Emperor of Ocean Park | 8.70 | Smokescreen (Eve Duncan, 25) | 0.68 |
| 5 | The Fire Thief (A Dark Paradise Mystery) | 8.69 | Crimes of Memory | 0.68 |

---

### Query 2 [E]: "cookbook recipes"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | The Ultimate Paleo Cookbook | 12.80 | 101 One-Dish Dinners | 0.38 |
| 2 | The Adventures of Fat Rice | 12.22 | FlavCity's 5 Ingredient Meals | 0.41 |
| 3 | The McDougall Quick and Easy Cookbook | 11.95 | Plated: Weeknight Dinners, Weekend Feasts | 0.41 |
| 4 | Cooking Know-How | 11.95 | Taste of Home Christmas 2014 | 0.44 |
| 5 | The Viennese Kitchen | 11.72 | Sunday Suppers | 0.45 |

---

### Query 3 [E]: "science fiction space"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | Ender's Game (The Ender Quintet) | 19.75 | Down and Out in the Magic Kingdom | 0.82 |
| 2 | I Am Crying All Inside (Clifford D. Simak) | 14.41 | Ender's Game (The Ender Quintet) | 0.97 |
| 3 | Stars Are Legion | 13.15 | Braking Day | 0.99 |
| 4 | The Garden of Rama | 13.06 | Protector of the Realm | 1.07 |
| 5 | Protector of the Realm | 12.81 | For Love of Mother-Not | 1.09 |

---

### Query 4 [E]: "python programming"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | The D Programming Language | 10.97 | Programming Arduino | 1.08 |
| 2 | Programming the World Wide Web | 10.88 | Genetic Algorithms and ML for Programmers | 1.12 |
| 3 | Definitive XML Application Development | 10.82 | Sqr in PeopleSoft and Other Applications | 1.21 |
| 4 | Programming Arduino | 10.02 | DKfindout! Coding | 1.25 |
| 5 | The Official BBC micro:bit User Guide | 9.70 | Beginner's Step-by-Step Coding Course | 1.25 |

---

### Query 5 [M]: "book to help with anxiety"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | Anxiety Relief for Teens | 23.97 | Everyone in This Room Will Someday Be Dead | 0.58 |
| 2 | Everyone in This Room Will Someday Be Dead | 22.75 | Anxiety Relief for Teens | 0.62 |
| 3 | Zendoodle Colorscapes: Outrageous Owls | 20.31 | God Gave Us You | 0.84 |
| 4 | Get the Guy | 18.96 | The Supreme Word Search Book for Adults | 0.89 |
| 5 | The Hoarder in You | 18.48 | Paying with Their Bodies | 0.90 |

---

### Query 6 [M]: "story about finding yourself"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | Eight Hundred Grapes: A Novel | 11.79 | The Way to Bea | 0.90 |
| 2 | The Whispering Soul | 11.48 | Don't Look for Me: A Novel | 0.96 |
| 3 | An Embarrassment of Witches | 11.14 | It Had To Be You | 1.00 |
| 4 | Spaghetti in a Hot Dog Bun | 10.57 | My Name Is Nathan Lucius | 1.01 |
| 5 | Distant Shores, Silent Thunder | 10.54 | Charleston: A Novel | 1.04 |

---

### Query 7 [M]: "guide for first time parents"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | Frommer's Alaska | 13.57 | Mac and Cheese (I Can Read Level 1) | 0.91 |
| 2 | The Jumbo Book of Art | 13.52 | Sleeping Bags To S'mores: Camping Basics | 0.91 |
| 3 | Lonely Planet Cruise Ports Alaska | 13.25 | The Best Seat in Second Grade | 0.92 |
| 4 | DK Eyewitness Italy: 2020 | 12.86 | Lessons Learned: The Kindergarten Survival Guide | 0.93 |
| 5 | Latin Primer 1 (Teacher Edition) | 12.70 | Regret Free Parenting | 0.94 |

---

### Query 8 [C]: "best book to learn machine learning with no math background"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | Genetic Algorithms and ML for Programmers | 28.12 | Genetic Algorithms and ML for Programmers | 0.92 |
| 2 | My Kindergarten Math Workbook | 25.42 | Pre-Algebra Essentials For Dummies | 0.94 |
| 3 | Pre-Algebra Essentials For Dummies | 24.23 | Bead Jewelry Making for Beginners | 0.97 |
| 4 | Piece in the Hoop | 23.92 | From the Moon, I Come in Peace | 1.01 |
| 5 | The Sewing Machine Accessory Bible | 23.74 | 101 Careers in Mathematics | 1.02 |

---

### Query 9 [C]: "historical fiction set in world war 2 from a female perspective"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | The Forgotten Room | 33.87 | The Women Who Flew for Hitler | 0.57 |
| 2 | Daughter of a Daughter of a Queen | 29.00 | Lilac Girls: A Novel | 0.59 |
| 3 | The Wind Is Not a River | 27.00 | The Alice Network | 0.64 |
| 4 | Age 14 | 26.32 | Dear Mrs. Bird: A Novel | 0.66 |
| 5 | The Long-Lost Secret Diary of the World's Worst Dinosaur Hunter | 25.93 | Dragon Harvest (The Lanny Budd Novels) | 0.69 |

---

### Query 10 [C]: "self help book for overcoming procrastination and building better habits"

| Rank | BM25 Title | BM25 Score | Semantic Title | Semantic Score |
|------|-----------|------------|----------------|----------------|
| 1 | Mini Habits for Teens | 24.77 | The 5-Minute Productivity Journal | 0.78 |
| 2 | The Fix: How Nations Survive and Thrive | 22.85 | Live With Purpose: Creating Positive, Lasting Change | 0.83 |
| 3 | Live With Purpose: Creating Positive, Lasting Change | 22.83 | The Man Who Wanted to Be Happy | 0.86 |
| 4 | The Man Who Wanted to Be Happy | 22.43 | If You Can Talk You Can Write | 0.94 |
| 5 | The Dad Connection | 22.24 | Five Acres and Independence | 0.95 |